In [ ]:
# Install: pip install pydantic openai pandas pillow requests fastapi uvicorn python-multipart

from pydantic import BaseModel, Field, field_validator, model_validator, HttpUrl
from typing import Optional, List, Literal
from enum import Enum
import base64
import json
import os
import re
from datetime import datetime
from io import BytesIO

from openai import OpenAI
from PIL import Image
import requests
import pandas as pd

# --- 1. SET UP OPENAI CLIENT ---
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

if not os.environ.get("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY is not set in your environment variables.")


# --- 2. SET UP PYDANTIC MODELS ---

class ProductCategory(str, Enum):
    """Valid product categories."""
    APPAREL = "Apparel"
    ACCESSORIES = "Accessories"
    FOOTWEAR = "Footwear"
    ELECTRONICS = "Electronics"
    HOME = "Home"
    BEAUTY = "Beauty"
    SPORTS = "Sports"
    OTHER = "Other"


class ProductGender(str, Enum):
    """Valid gender targets."""
    MEN = "Men"
    WOMEN = "Women"
    UNISEX = "Unisex"
    KIDS = "Kids"


class ProductSeason(str, Enum):
    """Valid seasons."""
    SPRING = "Spring"
    SUMMER = "Summer"
    FALL = "Fall"
    WINTER = "Winter"
    ALL_SEASON = "All Season"


class ProductInput(BaseModel):
    """
    Pydantic model for validating incoming product data.
    This is what clients send via JSON API requests.
    """
    
    # Required fields
    product_id: str = Field(
        ...,
        min_length=1,
        max_length=50,
        description="Unique product identifier"
    )
    
    product_name: str = Field(
        ...,
        min_length=3,
        max_length=200,
        description="Product display name"
    )
    
    # Price validation
    price: float = Field(
        ...,
        gt=0,
        le=100000,
        description="Product price (must be positive)"
    )
    
    currency: str = Field(
        default="USD",
        pattern=r"^[A-Z]{3}$",
        description="3-letter currency code"
    )
    
    # Category validation using enum
    category: ProductCategory = Field(
        ...,
        description="Product category"
    )
    
    # Optional fields with validation
    subcategory: Optional[str] = Field(
        default=None,
        max_length=100,
        description="Product subcategory"
    )
    
    gender: Optional[ProductGender] = Field(
        default=None,
        description="Target gender"
    )
    
    color: Optional[str] = Field(
        default=None,
        max_length=50,
        description="Primary color"
    )
    
    season: Optional[ProductSeason] = Field(
        default=None,
        description="Target season"
    )
    
    brand: Optional[str] = Field(
        default=None,
        max_length=100,
        description="Brand name"
    )
    
    # Image can be URL or base64
    image_url: Optional[HttpUrl] = Field(
        default=None,
        description="URL to product image"
    )
    
    image_base64: Optional[str] = Field(
        default=None,
        description="Base64 encoded image data"
    )
    
    # Additional metadata
    tags: Optional[List[str]] = Field(
        default=None,
        max_length=20,
        description="Product tags (max 20)"
    )
    
    description_hint: Optional[str] = Field(
        default=None,
        max_length=500,
        description="Optional hint for description generation"
    )

    # Field-level validators
    @field_validator('product_name')
    @classmethod
    def validate_product_name(cls, v: str) -> str:
        """Clean and validate product name."""
        v = v.strip()
        if not v:
            raise ValueError("Product name cannot be empty or whitespace")
        # Remove excessive whitespace
        v = re.sub(r'\s+', ' ', v)
        return v

    @field_validator('color')
    @classmethod
    def validate_color(cls, v: Optional[str]) -> Optional[str]:
        """Normalize color name."""
        if v is None:
            return None
        v = v.strip().title()
        return v

    @field_validator('tags')
    @classmethod
    def validate_tags(cls, v: Optional[List[str]]) -> Optional[List[str]]:
        """Clean and deduplicate tags."""
        if v is None:
            return None
        # Clean each tag
        cleaned = [tag.strip().lower() for tag in v if tag.strip()]
        # Remove duplicates while preserving order
        seen = set()
        unique = []
        for tag in cleaned:
            if tag not in seen:
                seen.add(tag)
                unique.append(tag)
        return unique

    @field_validator('image_base64')
    @classmethod
    def validate_base64(cls, v: Optional[str]) -> Optional[str]:
        """Validate base64 string format."""
        if v is None:
            return None
        # Remove data URL prefix if present
        if v.startswith('data:image'):
            v = v.split(',')[1] if ',' in v else v
        # Check if valid base64
        try:
            base64.b64decode(v)
        except Exception:
            raise ValueError("Invalid base64 encoding")
        return v

    # Model-level validator
    @model_validator(mode='after')
    def validate_image_source(self):
        """Ensure at least one image source is provided."""
        if self.image_url is None and self.image_base64 is None:
            raise ValueError("Either image_url or image_base64 must be provided")
        return self


class GeneratedListing(BaseModel):
    """
    Pydantic model for the generated product listing output.
    This is what the system returns after processing.
    """
    title: str = Field(..., description="Generated product title")
    description: str = Field(..., description="Generated product description")
    features: List[str] = Field(default_factory=list, description="Product features")
    tags: List[str] = Field(default_factory=list, description="SEO tags")
    detected_attributes: dict = Field(default_factory=dict, description="AI-detected attributes")


class ProductListingResponse(BaseModel):
    """
    Complete API response model.
    """
    success: bool
    product_id: str
    original_input: ProductInput
    generated_listing: Optional[GeneratedListing] = None
    error_message: Optional[str] = None
    processing_time_seconds: float
    timestamp: str = Field(default_factory=lambda: datetime.utcnow().isoformat())


class BatchProductRequest(BaseModel):
    """
    Model for batch processing multiple products.
    """
    products: List[ProductInput] = Field(
        ...,
        min_length=1,
        max_length=50,
        description="List of products to process (max 50)"
    )
    
    priority: Literal["low", "normal", "high"] = Field(
        default="normal",
        description="Processing priority"
    )


class ValidationErrorDetail(BaseModel):
    """
    Detailed validation error response.
    """
    field: str
    message: str
    invalid_value: Optional[str] = None


class ValidationErrorResponse(BaseModel):
    """
    Response model for validation errors.
    """
    success: bool = False
    error_type: str = "validation_error"
    error_count: int
    errors: List[ValidationErrorDetail]
    timestamp: str = Field(default_factory=lambda: datetime.utcnow().isoformat())


# --- 3. VALIDATION AND ERROR HANDLING ---

def validate_product_input(json_data: dict) -> tuple[Optional[ProductInput], Optional[ValidationErrorResponse]]:
    """
    Validate incoming JSON data and return either valid product or error response.
    """
    try:
        product = ProductInput(**json_data)
        return product, None
    
    except Exception as e:
        errors = []
        
        if hasattr(e, 'errors'):
            # Pydantic validation errors
            for error in e.errors():
                field = '.'.join(str(loc) for loc in error['loc'])
                errors.append(ValidationErrorDetail(
                    field=field,
                    message=error['msg'],
                    invalid_value=str(error.get('input', ''))[:100]
                ))
        else:
            # Other errors
            errors.append(ValidationErrorDetail(
                field="unknown",
                message=str(e)
            ))
        
        error_response = ValidationErrorResponse(
            error_count=len(errors),
            errors=errors
        )
        
        return None, error_response


def validate_batch_request(json_data: dict) -> tuple[Optional[BatchProductRequest], Optional[ValidationErrorResponse]]:
    """
    Validate batch product request.
    """
    try:
        batch = BatchProductRequest(**json_data)
        return batch, None
    
    except Exception as e:
        errors = []
        
        if hasattr(e, 'errors'):
            for error in e.errors():
                field = '.'.join(str(loc) for loc in error['loc'])
                errors.append(ValidationErrorDetail(
                    field=field,
                    message=error['msg'],
                    invalid_value=str(error.get('input', ''))[:100]
                ))
        else:
            errors.append(ValidationErrorDetail(
                field="unknown",
                message=str(e)
            ))
        
        error_response = ValidationErrorResponse(
            error_count=len(errors),
            errors=errors
        )
        
        return None, error_response


# --- 4. IMAGE PROCESSING ---

def get_image_base64_from_product(product: ProductInput) -> str:
    """
    Get base64 image from validated product input.
    Handles both URL and base64 input.
    """
    if product.image_base64:
        return product.image_base64
    
    if product.image_url:
        try:
            response = requests.get(str(product.image_url), timeout=15)
            response.raise_for_status()
            image = Image.open(BytesIO(response.content)).convert("RGB")
            
            buffer = BytesIO()
            image.save(buffer, format="JPEG")
            return base64.b64encode(buffer.getvalue()).decode("utf-8")
        
        except Exception as e:
            raise ValueError(f"Failed to download image from URL: {e}")
    
    raise ValueError("No valid image source available")


# --- 5. OPENAI INTEGRATION ---

def generate_listing_from_validated_product(product: ProductInput) -> GeneratedListing:
    """
    Generate product listing using OpenAI vision from validated product data.
    """
    image_b64 = get_image_base64_from_product(product)
    
    prompt = f"""
Analyze this product image and metadata, then generate a compelling e-commerce listing.

Product Information:
- Name: {product.product_name}
- Category: {product.category.value}
- Subcategory: {product.subcategory or 'N/A'}
- Brand: {product.brand or 'N/A'}
- Color: {product.color or 'N/A'}
- Gender: {product.gender.value if product.gender else 'N/A'}
- Season: {product.season.value if product.season else 'N/A'}
- Price: {product.price} {product.currency}
- Tags: {', '.join(product.tags) if product.tags else 'N/A'}
- Description hint: {product.description_hint or 'N/A'}

Instructions:
1. Create an attractive, SEO-friendly title
2. Write a compelling description (80-150 words)
3. List 5 key features as bullet points
4. Suggest 8-12 relevant tags
5. Identify visible product attributes

Return ONLY valid JSON in this exact format:
{{
    "title": "string",
    "description": "string",
    "features": ["feature1", "feature2", "feature3", "feature4", "feature5"],
    "tags": ["tag1", "tag2", "tag3"],
    "detected_attributes": {{
        "product_type": "string",
        "color": "string",
        "style": "string",
        "material": "string if visible",
        "audience": "string"
    }}
}}
"""

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role": "system",
                "content": "You are an expert e-commerce copywriter. Return only valid JSON."
            },
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{image_b64}"
                        }
                    }
                ]
            }
        ],
        max_tokens=800,
        temperature=0.4
    )

    content = response.choices[0].message.content
    
    # Clean markdown formatting if present
    cleaned = content.strip()
    if cleaned.startswith("```json"):
        cleaned = cleaned[7:]
    if cleaned.startswith("```"):
        cleaned = cleaned[3:]
    if cleaned.endswith("```"):
        cleaned = cleaned[:-3]
    cleaned = cleaned.strip()
    
    parsed = json.loads(cleaned)
    
    return GeneratedListing(**parsed)


# --- 6. END-TO-END PROCESSING ---

def process_product_request(json_data: dict) -> ProductListingResponse:
    """
    Process a single product request end-to-end.
    Validates input, generates listing, returns structured response.
    """
    import time
    start_time = time.time()
    
    # Step 1: Validate input
    product, validation_error = validate_product_input(json_data)
    
    if validation_error:
        return ProductListingResponse(
            success=False,
            product_id=json_data.get('product_id', 'unknown'),
            original_input=None,
            error_message=f"Validation failed: {validation_error.model_dump_json()}",
            processing_time_seconds=time.time() - start_time
        )
    
    # Step 2: Generate listing
    try:
        listing = generate_listing_from_validated_product(product)
        
        return ProductListingResponse(
            success=True,
            product_id=product.product_id,
            original_input=product,
            generated_listing=listing,
            processing_time_seconds=time.time() - start_time
        )
    
    except Exception as e:
        return ProductListingResponse(
            success=False,
            product_id=product.product_id,
            original_input=product,
            error_message=f"Processing failed: {str(e)}",
            processing_time_seconds=time.time() - start_time
        )


def process_batch_request(json_data: dict) -> List[ProductListingResponse]:
    """
    Process multiple products in a batch.
    """
    batch, validation_error = validate_batch_request(json_data)
    
    if validation_error:
        return [{
            "success": False,
            "error": validation_error.model_dump()
        }]
    
    results = []
    for product in batch.products:
        result = process_product_request(product.model_dump())
        results.append(result)
    
    return results


# --- 7. DEMONSTRATION ---

def demo_validation():
    """
    Demonstrate the validation system with various test cases.
    """
    print("=" * 60)
    print("PYDANTIC VALIDATION DEMONSTRATION")
    print("=" * 60)
    
    # Test Case 1: Valid product
    print("\n--- Test 1: Valid Product ---")
    valid_product = {
        "product_id": "PROD-001",
        "product_name": "Premium Cotton T-Shirt",
        "price": 29.99,
        "currency": "USD",
        "category": "Apparel",
        "subcategory": "T-Shirts",
        "gender": "Men",
        "color": "navy blue",
        "season": "Summer",
        "brand": "StyleCo",
        "image_url": "https://example.com/images/tshirt.jpg",
        "tags": ["casual", "cotton", "comfortable", "everyday"],
        "description_hint": "Focus on breathability and comfort"
    }
    
    product, error = validate_product_input(valid_product)
    if product:
        print("✓ Validation passed!")
        print(f"  Product ID: {product.product_id}")
        print(f"  Name: {product.product_name}")
        print(f"  Price: {product.price} {product.currency}")
        print(f"  Category: {product.category.value}")
        print(f"  Color (normalized): {product.color}")
        print(f"  Tags (cleaned): {product.tags}")
    else:
        print(f"✗ Validation failed: {error}")
    
    # Test Case 2: Invalid price
    print("\n--- Test 2: Invalid Price (negative) ---")
    invalid_price = {
        "product_id": "PROD-002",
        "product_name": "Test Product",
        "price": -10.00,
        "category": "Apparel",
        "image_url": "https://example.com/image.jpg"
    }
    
    product, error = validate_product_input(invalid_price)
    if error:
        print("✓ Validation correctly rejected invalid data!")
        for err in error.errors:
            print(f"  Field: {err.field}")
            print(f"  Message: {err.message}")
    
    # Test Case 3: Missing required field
    print("\n--- Test 3: Missing Required Field ---")
    missing_field = {
        "product_id": "PROD-003",
        "price": 19.99,
        "category": "Apparel",
        "image_url": "https://example.com/image.jpg"
    }
    
    product, error = validate_product_input(missing_field)
    if error:
        print("✓ Validation correctly caught missing field!")
        for err in error.errors:
            print(f"  Field: {err.field}")
            print(f"  Message: {err.message}")
    
    # Test Case 4: Invalid category
    print("\n--- Test 4: Invalid Category ---")
    invalid_category = {
        "product_id": "PROD-004",
        "product_name": "Test Product",
        "price": 29.99,
        "category": "InvalidCategory",
        "image_url": "https://example.com/image.jpg"
    }
    
    product, error = validate_product_input(invalid_category)
    if error:
        print("✓ Validation correctly rejected invalid category!")
        for err in error.errors:
            print(f"  Field: {err.field}")
            print(f"  Message: {err.message}")
    
    # Test Case 5: No image provided
    print("\n--- Test 5: No Image Source ---")
    no_image = {
        "product_id": "PROD-005",
        "product_name": "Product Without Image",
        "price": 49.99,
        "category": "Electronics"
    }
    
    product, error = validate_product_input(no_image)
    if error:
        print("✓ Validation correctly requires image!")
        for err in error.errors:
            print(f"  Field: {err.field}")
            print(f"  Message: {err.message}")
    
    # Test Case 6: Invalid currency code
    print("\n--- Test 6: Invalid Currency Code ---")
    invalid_currency = {
        "product_id": "PROD-006",
        "product_name": "Test Product",
        "price": 29.99,
        "currency": "INVALID",
        "category": "Apparel",
        "image_url": "https://example.com/image.jpg"
    }
    
    product, error = validate_product_input(invalid_currency)
    if error:
        print("✓ Validation correctly rejected invalid currency!")
        for err in error.errors:
            print(f"  Field: {err.field}")
            print(f"  Message: {err.message}")
    
    # Test Case 7: Product name too short
    print("\n--- Test 7: Product Name Too Short ---")
    short_name = {
        "product_id": "PROD-007",
        "product_name": "AB",
        "price": 29.99,
        "category": "Apparel",
        "image_url": "https://example.com/image.jpg"
    }
    
    product, error = validate_product_input(short_name)
    if error:
        print("✓ Validation correctly rejected short name!")
        for err in error.errors:
            print(f"  Field: {err.field}")
            print(f"  Message: {err.message}")
    
    print("\n" + "=" * 60)
    print("VALIDATION DEMONSTRATION COMPLETE")
    print("=" * 60)


def demo_end_to_end_processing():
    """
    Demonstrate end-to-end processing with a real product.
    Note: This requires a valid image URL and API key.
    """
    print("\n" + "=" * 60)
    print("END-TO-END PROCESSING DEMONSTRATION")
    print("=" * 60)
    
    # Example product with a real image URL
    # Replace with an actual accessible image URL for testing
    test_product = {
        "product_id": "DEMO-001",
        "product_name": "Classic Blue Denim Jacket",
        "price": 89.99,
        "currency": "USD",
        "category": "Apparel",
        "subcategory": "Jackets",
        "gender": "Unisex",
        "color": "Blue",
        "season": "Fall",
        "brand": "DenimCo",
        "image_url": "https://images.unsplash.com/photo-1576995853123-5a10305d93c0?w=400",
        "tags": ["denim", "casual", "classic", "layering"],
        "description_hint": "Versatile everyday jacket"
    }
    
    print("\nProcessing product request...")
    print(f"Product: {test_product['product_name']}")
    
    result = process_product_request(test_product)
    
    if result.success:
        print("\n✓ Processing successful!")
        print(f"  Processing time: {result.processing_time_seconds:.2f}s")
        print(f"\n  Generated Title: {result.generated_listing.title}")
        print(f"\n  Description: {result.generated_listing.description[:200]}...")
        print(f"\n  Features:")
        for feature in result.generated_listing.features:
            print(f"    • {feature}")
        print(f"\n  Tags: {', '.join(result.generated_listing.tags)}")
    else:
        print(f"\n✗ Processing failed: {result.error_message}")
    
    print("\n" + "=" * 60)


# --- 8. FASTAPI INTEGRATION (Optional) ---

def create_fastapi_app():
    """
    Create a FastAPI application with Pydantic validation.
    Run with: uvicorn script_name:app --reload
    """
    from fastapi import FastAPI, HTTPException
    from fastapi.responses import JSONResponse
    
    app = FastAPI(
        title="Product Listing Generator API",
        description="Generate AI-powered product listings with Pydantic validation",
        version="1.0.0"
    )
    
    @app.post("/api/v1/product/generate", response_model=ProductListingResponse)
    async def generate_listing(product: ProductInput):
        """
        Generate a product listing from validated input.
        """
        result = process_product_request(product.model_dump())
        
        if not result.success:
            raise HTTPException(status_code=400, detail=result.error_message)
        
        return result
    
    @app.post("/api/v1/product/validate")
    async def validate_product(product_data: dict):
        """
        Validate product data without generating listing.
        """
        product, error = validate_product_input(product_data)
        
        if error:
            return JSONResponse(
                status_code=400,
                content=error.model_dump()
            )
        
        return {
            "success": True,
            "message": "Validation passed",
            "validated_data": product.model_dump()
        }
    
    @app.post("/api/v1/product/batch")
    async def batch_generate(batch: BatchProductRequest):
        """
        Generate listings for multiple products.
        """
        results = process_batch_request(batch.model_dump())
        return {"results": [r.model_dump() if hasattr(r, 'model_dump') else r for r in results]}
    
    return app


# Create app instance for uvicorn
# Uncomment the line below when running as API server
# app = create_fastapi_app()


# --- 9. RUN DEMONSTRATION ---

if __name__ == "__main__":
    # Run validation demonstration
    demo_validation()
    
    # Uncomment below to test end-to-end processing
    # (requires valid API key and accessible image URL)
    # demo_end_to_end_processing()
    
    print("\n" + "=" * 60)
    print("TO RUN AS API SERVER:")
    print("1. Uncomment 'app = create_fastapi_app()' line")
    print("2. Run: uvicorn filename:app --reload")
    print("3. Visit: http://localhost:8000/docs for API documentation")
    print("=" * 60)

PYDANTIC VALIDATION DEMONSTRATION

--- Test 1: Valid Product ---
✓ Validation passed!
  Product ID: PROD-001
  Name: Premium Cotton T-Shirt
  Price: 29.99 USD
  Category: Apparel
  Color (normalized): Navy Blue
  Tags (cleaned): ['casual', 'cotton', 'comfortable', 'everyday']

--- Test 2: Invalid Price (negative) ---
✓ Validation correctly rejected invalid data!
  Field: price
  Message: Input should be greater than 0

--- Test 3: Missing Required Field ---
✓ Validation correctly caught missing field!
  Field: product_name
  Message: Field required

--- Test 4: Invalid Category ---
✓ Validation correctly rejected invalid category!
  Field: category
  Message: Input should be 'Apparel', 'Accessories', 'Footwear', 'Electronics', 'Home', 'Beauty', 'Sports' or 'Other'

--- Test 5: No Image Source ---
✓ Validation correctly requires image!
  Field: 
  Message: Value error, Either image_url or image_base64 must be provided

--- Test 6: Invalid Currency Code ---
✓ Validation correctly rejecte

C:\Users\Adetunji\AppData\Local\Temp\ipykernel_17560\643995967.py:272: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp: str = Field(default_factory=lambda: datetime.utcnow().isoformat())
